<a href="https://colab.research.google.com/github/Shannonfireball/healthCostsPrediction/blob/main/fcc_predict_health_costs_with_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
train_dataset_df = dataset.sample( frac=0.8, random_state=0 )
test_dataset_df = dataset.drop(train_dataset_df.index)

train_lables = train_dataset_df.pop('expenses')
test_lables = test_dataset_df.pop('expenses')
print("train_lables",train_lables)
print("test_lables",test_lables)

In [ ]:


categoricalColumns = train_dataset_df.select_dtypes(['object']).columns

for column in categoricalColumns:
  lables = train_dataset_df[column].astype('category').cat.categories
  for code, name in enumerate(lables):
    print(f'code: {code}, name: {name}')
  train_dataset_df[column] = train_dataset_df[column].astype('category').cat.codes
  test_dataset_df[column] = test_dataset_df[column].astype('category').cat.codes


normalizer = layers.Normalization(axis=-1)
normalizer.adapt(train_dataset_df.values)

model = keras.Sequential([
    normalizer,
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_absolute_error',
    metrics=['mae','mse']
)

history = model.fit(
    train_dataset_df,
    train_lables,
    epochs=1000,
    validation_split=0.2,
    verbose=0,
    callbacks=[tfdocs.modeling.EpochDots()]
)

In [ ]:
test_dataset = test_dataset_df
test_labels = test_lables

print(len(test_dataset))
print(len(test_labels))

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
